## Import Libraries

In [ ]:
# Web scraping
from bs4 import BeautifulSoup, NavigableString
import requests
import urllib.parse # ASCII hex code

# Store data
import pandas as pd
import re # string manipulation/find patterns

## Get the LinkedIn URL

In [ ]:
# Generate a url based on job title and job location (one page)
def get_job(job_title, job_location):
    clean_url = "https://www.linkedin.com/jobs/search?keywords="
    clean_url += urllib.parse.quote(job_title)
    clean_url += "&location="
    clean_url += urllib.parse.quote(job_location)
    clean_url += "&geoId=105517665&trk=public_jobs_jobs-search-bar_search-submit&position=1&pageNum=0"
    
    return clean_url

In [4]:
data_analyst_url = get_job("Data Analyst", "Tampa FL, United States")

In [ ]:
# Check if url works
requests.get(data_analyst_url)

<Response [200]>

In [ ]:
# Create an soup object to hold the html content
page = requests.get(data_analyst_url)
soup = BeautifulSoup(page.text, 'html.parser')

## Create DataFrame columns

Create lists that represent different aspects of the job

In [10]:
job_titles = []
company_names = []
company_linkedins = []
job_locations = []
dates_posted = []
job_levels = []
employment_types = []
employment_levels = []
salaries = []

## Extract Job Data

In [ ]:
# scrape basic job information
jobs = soup.find_all(class_="base-search-card__info")

# scrape each job card
for job in jobs:
    job_title = job.find('h3').text.strip()
    company_name = job.find('h4').text.strip()
    linkedin_link = job.find('a') # don't add to df
    company_linkedin = linkedin_link.get('href')
    job_metadata = job.find(class_="base-search-card__metadata") # don't add to df
    job_location = job_metadata.find(class_="job-search-card__location").text.strip()
    date_posted = job.find('time').get('datetime')

    job_titles.append(job_title)
    company_names.append(company_name)
    company_linkedins.append(company_linkedin)
    job_locations.append(job_location)
    dates_posted.append(date_posted)
    

In [ ]:
# scrape job descriptions/navigate to main description page
job_descriptions = soup.find_all(class_="base-card__full-link absolute top-0 right-0 bottom-0 left-0 p-0 z-[2] outline-offset-[4px]")
for job_description in job_descriptions:
    job_description_link = job_description.get('href')
    job_description_page = requests.get(job_description_link)
    
    # individual job description
    page_soup = BeautifulSoup(job_description_page.text, 'html.parser')
    attributes = page_soup.find_all(class_="description__job-criteria-item")
    for attribute in attributes:
        # get job level
        tag = attribute.find('span')
        if tag.parent.find('h3').text.strip() == "Seniority level":
            employment_levels.append(tag.text.strip())
        # get employment type
        if tag.parent.find('h3').text.strip() == "Employment type":
            employment_types.append(tag.text.strip())
    
    # Check if salary is present
    sal = False
    for node in page_soup.descendants:
        if isinstance(node, NavigableString):
            if node.find_parent("section", class_="core-section-container my-3 description"):
                if "$" in node.text.strip():
                    salary = node.text.strip()
                    salaries.append(salary)
                    sal = True
                    break
    if sal == False:
        salaries.append('N/A')


$76 560,00 - $106 840,00
The pay range for this position is $25.00 - $28.00/hr.
The pay range for this position is $25.00 - $28.00/hr.
Baker Tilly Advisory Group, LP and Baker Tilly US, LLP, trading as Baker Tilly, are independent members of Baker Tilly International, a worldwide network of independent accounting and business advisory firms in 141 territories, with 43,000 professionals and a combined worldwide revenue of $5.2 billion. Visit bakertilly.com or join the conversation on LinkedIn, Facebook and Instagram.
Kforce in Tampa, Florida is looking for Lead Analysts. Qualified candidates will be planning and executing a variety of methodologies as part of the concept stage in the overall project development of complex business intelligence tools or data warehouse and systems; Designing, building, and rolling out of high performing business intelligence tools, including design of related master data management sets and maps for mining in a cloud (AWS/GCP) environment; Researching, pl

## Verify Consistent Column Length

DataFrame must have consistent column length

In [23]:
columns = [
    job_titles,
    company_names,
    company_linkedins,
    job_locations,
    dates_posted,
    employment_types,
    employment_levels,
    salaries
]

for column in columns:
    print(len(column))

57
57
57
57
57
57
57
57


## Make the DataFrame

In [14]:
job_df = pd.DataFrame({
    'job_title' : job_titles,
    'company_name' : company_names,
    'company_linkedin' : company_linkedins,
    'job_location': job_locations,
    'date_posted' : dates_posted,
    'employment_type': employment_types,
    'employment_level': employment_levels,
    'salary': salaries
})

In [15]:
job_df.head(10)

,job_title,company_name,company_linkedin,job_location,date_posted,employment_type,employment_level,salary
0,Data Analyst,Gerdau North America,https://www.linkedin.com/company/gerdau-north-...,"Tampa, FL",2026-04-16,Full-time,Entry level,N/A
1,Data Analyst,Citi,https://www.linkedin.com/company/citi?trk=publ...,"Tampa, FL",2026-04-10,Full-time,Not Applicable,"$76 560,00 - $106 840,00"
2,Data Analyst,Insight Global,https://www.linkedin.com/company/insight-globa...,"Clearwater, FL",2026-04-23,Contract,Mid-Senior level,N/A
3,Data Analyst,TEKsystems,https://www.linkedin.com/company/teksystems?tr...,"Tampa, FL",2026-04-22,Full-time,Entry level,The pay range for this position is $25.00 - $2...
4,Data Analyst,TEKsystems,https://www.linkedin.com/company/teksystems?tr...,"Tampa, FL",2026-04-23,Full-time,Entry level,The pay range for this position is $25.00 - $2...
5,Data Analyst,Fintech,https://www.linkedin.com/company/fintech-net?t...,"Tampa, FL",2026-04-14,Full-time,Mid-Senior level,N/A
6,Data Analytics Associate,Baker Tilly US,https://www.linkedin.com/company/bakertillyus?...,Greater Tampa Bay Area,2026-04-23,Full-time,Entry level,"Baker Tilly Advisory Group, LP and Baker Tilly..."
7,Lead Analysts,Kforce Inc,https://www.linkedin.com/company/kforce?trk=pu...,"Tampa, FL",2026-04-21,Contract,Associate,"Kforce in Tampa, Florida is looking for Lead A..."
8,Data Analyst,General Dynamics Information Technology,https://www.linkedin.com/company/gdit?trk=publ...,"Tampa, FL",2026-04-18,Full-time,Mid-Senior level,The likely salary range for this position is $...
9,Jr Data Analyst - Tampa 1001215,Jobs via Dice,https://www.linkedin.com/company/jobs-via-dice...,"Tampa, FL",2026-04-23,Full-time,Mid-Senior level,N/A


In [16]:
job_df.shape

(57, 8)

## Clean the salary column

Get annual and per hour (single and range). Can potentially normalize further.

In [17]:
def normalize_salary(match):
    match_string = match.group(0).lower()

    match_string = match_string.replace('–','-')
    match_string = match_string.replace(' - ', '-')
    match_string = match_string.replace('k',',000')
    match_string = match_string.replace(' to ','-')
    match_string = match_string.replace('$ ', '$')

    # clean trailing characters
    match_string = match_string.rstrip().rstrip(".")

    if "-" in match_string and " " in match_string:
        match_string = match_string.replace(',', '.')
        match_string = match_string.replace(' ', ',')
    
    # remove decimals
    match_string = match_string.replace('.00', '')

    return match_string

def clean_salary(row):
    hourly_single_pattern = r"\$?\s*([\d][\d.,]+)\s*(?:/hr|/hour|per hour|an hour)"
    hourly_range_pattern = r"\$?\s*([\d][\d.,]+)\s*(?:[-–]|to)\s*([\d][\d.,]+)\s*(?:/hr|/hour|per hour|an hour)"
    annual_range_pattern = r"\$?\s*([\d][\d\s.,]*k?)\s*(?:[-–]|to)\s*\$?\s*([\d][\d\s.,]*k?)"

    hourly_single_match = re.search(hourly_single_pattern, row, flags=re.I)
    hourly_range_match = re.search(hourly_range_pattern, row, flags=re.I)
    annual_range_match = re.search(annual_range_pattern, row, flags=re.I)

    if hourly_range_match:
        hourly_range_string = normalize_salary(hourly_range_match)
        return hourly_range_string
    elif hourly_single_match:
        hourly_string = normalize_salary(hourly_single_match)
        return hourly_string
    elif annual_range_match:
        range_string = normalize_salary(annual_range_match)
        return range_string
    else:
        return "N/A"


job_df['salary'] = job_df['salary'].apply(clean_salary)
job_df.head()


,job_title,company_name,company_linkedin,job_location,date_posted,employment_type,employment_level,salary
0,Data Analyst,Gerdau North America,https://www.linkedin.com/company/gerdau-north-...,"Tampa, FL",2026-04-16,Full-time,Entry level,N/A
1,Data Analyst,Citi,https://www.linkedin.com/company/citi?trk=publ...,"Tampa, FL",2026-04-10,Full-time,Not Applicable,"$76,560-$106,840"
2,Data Analyst,Insight Global,https://www.linkedin.com/company/insight-globa...,"Clearwater, FL",2026-04-23,Contract,Mid-Senior level,N/A
3,Data Analyst,TEKsystems,https://www.linkedin.com/company/teksystems?tr...,"Tampa, FL",2026-04-22,Full-time,Entry level,$28/hr
4,Data Analyst,TEKsystems,https://www.linkedin.com/company/teksystems?tr...,"Tampa, FL",2026-04-23,Full-time,Entry level,$28/hr


## Clean Employment Level

In [18]:
job_df['employment_level'] = job_df['employment_level'].apply(lambda x : "N/A" if x in "Not Applicable" else x)
job_df['employment_level'].unique()

<StringArray>
['Entry level', 'N/A', 'Mid-Senior level', 'Associate', 'Internship',
 'Director']
Length: 6, dtype: str

## Save to CSV

In [19]:
job_df.to_csv('da_jobs.csv', index=False)
print("Successfuly saved.")

Successfuly saved.
